# Lunar Tear Patcher (JP)

Patch a NieR Re\[in\]carnation **JP** APK and master data for use with a private server.

**How to use:**
1. Fill in the **Configuration** form below.
2. Run cells top-to-bottom, or use **Runtime → Run all**.
3. Patched files will download to your browser.

**Two modes:**
- **Drive mode** (default): Check **use_gdrive**, enter paths relative to `/content/drive/` (e.g. `MyDrive/original.apk`).
- **URL mode**: Uncheck **use_gdrive**, paste download URLs into `apk_source` / `masterdata_source`.

Each section is independent — skip what you don't need.


In [ ]:
#@title Configuration

#@markdown ### Source Mode
use_gdrive = True #@param {type:"boolean"}

#@markdown ---
#@markdown ### File Sources
#@markdown With **use_gdrive** checked, enter Drive paths relative to `/content/drive/`
#@markdown (e.g. `MyDrive/original.apk`). Otherwise, enter download URLs.
apk_source = "MyDrive/NieR Re[in]carnation_3.7.1_JP.apk" #@param {type:"string"}
masterdata_source = "MyDrive/20240404193219.bin.e" #@param {type:"string"}

#@markdown ---
#@markdown ### Server Settings (JP)
grpc_addr = "ip:8003" #@param {type:"string"}
http_addr = "ip:8080" #@param {type:"string"}
auth_host = "ip:3000" #@param {type:"string"}
scripts_branch = "main" #@param {type:"string"}

import os

if use_gdrive:
    from google.colab import drive
    drive.mount('/content/drive')
    print('Google Drive mounted at /content/drive\n')
    for label, src in [('APK', apk_source), ('Master data', masterdata_source)]:
        if not src:
            print(f'  {label}: (not set, will skip)')
            continue
        path = os.path.join('/content/drive', src)
        if os.path.isfile(path):
            size = os.path.getsize(path)
            print(f'  {label}: {path} — OK ({size / 1024:.1f} KB)')
        else:
            print(f'  {label}: {path} — NOT FOUND')
else:
    print('URL mode — files will be downloaded during patching.')


In [ ]:
#@title Install dependencies
import shutil, os

if not os.path.isdir('/content/scripts-jp'):
    !git clone --depth 1 --branch {scripts_branch} https://github.com/lovelyzy7/lunar-scripts-jp.git /content/scripts-jp
else:
    !cd /content/scripts-jp && git fetch origin {scripts_branch} && git checkout {scripts_branch} && git pull origin {scripts_branch}
    print(f'Scripts updated to branch: {scripts_branch}')

# Fail fast: never continue with missing scripts (private/deleted repo, wrong branch, no network)
_REQUIRED = ['requirements.txt', 'android/patch_apk_jp.py', 'masterdata/patch_masterdata.py']
_missing = [f for f in _REQUIRED if not os.path.isfile(os.path.join('/content/scripts-jp', f))]
if _missing:
    raise RuntimeError(
        'Patch scripts missing from /content/scripts-jp: ' + ', '.join(_missing) + '\n'
        'The git clone above probably failed (repo private / not found / wrong branch / no network). '
        'Fix that, then re-run this cell.'
    )

!pip install -q -r /content/scripts-jp/requirements.txt

!apt-get -qq update > /dev/null 2>&1

# apktool — install from GitHub for a recent version
if not shutil.which('apktool'):
    !wget -q -O /usr/local/bin/apktool.jar \
      https://github.com/iBotPeaches/Apktool/releases/download/v2.11.1/apktool_2.11.1.jar
    !wget -q -O /usr/local/bin/apktool \
      https://raw.githubusercontent.com/iBotPeaches/Apktool/master/scripts/linux/apktool
    !chmod +x /usr/local/bin/apktool

# zipalign + apksigner
if not shutil.which('zipalign') or not shutil.which('apksigner'):
    !apt-get -qq install -y zipalign apksigner > /dev/null 2>&1

for tool in ['apktool', 'zipalign', 'apksigner', 'keytool']:
    path = shutil.which(tool)
    status = path or 'NOT FOUND'
    print(f'  {tool}: {status}')

print('\nDone.')


In [ ]:
#@title Patch APK
import urllib.request, shutil, os, subprocess, sys
from google.colab import files

def _run(cmd, label):
    if subprocess.run(cmd, shell=True).returncode != 0:
        raise RuntimeError(f'{label} failed — see the output above.')

assert os.path.isfile('/content/scripts-jp/android/patch_apk_jp.py'), \
    'Patch scripts not found — run the "Install dependencies" cell first.'
assert apk_source, 'Set apk_source in the Configuration cell first.'
assert grpc_addr, 'Set grpc_addr in the Configuration cell first.'
assert http_addr, 'Set http_addr in the Configuration cell first.'

if use_gdrive:
    apk_path = os.path.join('/content/drive', apk_source)
    assert os.path.isfile(apk_path), f'File not found: {apk_path}'
    shutil.copy2(apk_path, '/content/original.apk')
    size_mb = os.path.getsize('/content/original.apk') / 1024 / 1024
    print(f'Copied APK from Drive: {apk_path} ({size_mb:.1f} MB)')
else:
    print(f'Downloading APK from {apk_source} ...')
    urllib.request.urlretrieve(apk_source, '/content/original.apk')
    size_mb = os.path.getsize('/content/original.apk') / 1024 / 1024
    print(f'  Downloaded {size_mb:.1f} MB')

print('\nDecompiling ...')
_run('apktool d /content/original.apk -o /content/patched -f', 'apktool decompile')

auth_flag = f'--auth-host {auth_host}' if auth_host else ''
print('\nPatching ...')
_run(f'{sys.executable} /content/scripts-jp/android/patch_apk_jp.py /content/patched --grpc-addr {grpc_addr} --http-addr {http_addr} {auth_flag}',
     'patch_apk_jp')

print('\nRebuilding ...')
_run('apktool b /content/patched -o /content/patched_unaligned.apk', 'apktool rebuild')

print('\nAligning ...')
_run('zipalign -p -f 4 /content/patched_unaligned.apk /content/patched-jp.apk', 'zipalign')

# Unified signing: same key as local build_apk.py / README (ships with the repo)
KEYSTORE = '/content/scripts-jp/keys/nier-jp.keystore'
if not os.path.isfile(KEYSTORE):
    raise RuntimeError('Keystore not found: ' + KEYSTORE +
                       ' — it ships with the repo; re-run "Install dependencies".')

print('\nSigning ...')
_run(f'apksigner sign --ks {KEYSTORE} --ks-pass pass:lunar-jp --ks-key-alias nier '
     f'--key-pass pass:lunar-jp /content/patched-jp.apk', 'apksigner')

size_mb = os.path.getsize('/content/patched-jp.apk') / 1024 / 1024
print(f'\nDone! Patched APK: {size_mb:.1f} MB')

#files.download('/content/patched-jp.apk')


In [ ]:
#@title Save to Google Drive
import shutil
import os

#@markdown ### Patched APK
output_drive_path = "MyDrive/patched-jp.apk" #@param {type:"string"}

def save_to_drive(src, dst, label):
    if not dst:
        print(f"{label}: skipped (path not set)")
        return
    if not os.path.exists(src):
        print(f"{label}: not found at {src} — ensure the previous cell ran successfully.")
        return
    destination_path = os.path.join('/content/drive', dst)
    destination_dir = os.path.dirname(destination_path)
    if destination_dir and not os.path.exists(destination_dir):
        print(f"Creating directory: {destination_dir}")
        os.makedirs(destination_dir)
    try:
        shutil.copy2(src, destination_path)
        size_mb = os.path.getsize(destination_path) / 1024 / 1024
        print(f"{label}: successfully copied to Google Drive: {destination_path} ({size_mb:.1f} MB)")
    except Exception as e:
        print(f"{label}: error copying file to Google Drive: {e}")

if not use_gdrive:
    print("Google Drive is not configured. Please enable 'use_gdrive' in the Configuration cell if you wish to save to Drive.")
elif not os.path.isdir('/content/drive'):
    print("Google Drive is not mounted. Please ensure it's mounted correctly.")
else:
    save_to_drive('/content/patched-jp.apk', output_drive_path, 'Patched APK')


In [ ]:
#@title Patch Master Data
import urllib.request, shutil, os, subprocess, sys
from google.colab import files

def _run(cmd, label):
    if subprocess.run(cmd, shell=True).returncode != 0:
        raise RuntimeError(f'{label} failed — see the output above.')

assert os.path.isfile('/content/scripts-jp/masterdata/patch_masterdata.py'), \
    'Patch scripts not found — run the "Install dependencies" cell first.'
assert masterdata_source, 'Set masterdata_source in the Configuration cell first.'

if use_gdrive:
    md_path = os.path.join('/content/drive', masterdata_source)
    assert os.path.isfile(md_path), f'File not found: {md_path}'
    shutil.copy2(md_path, '/content/masterdata.bin.e')
    size_kb = os.path.getsize('/content/masterdata.bin.e') / 1024
    print(f'Copied master data from Drive: {md_path} ({size_kb:.1f} KB)')
else:
    print(f'Downloading master data from {masterdata_source} ...')
    urllib.request.urlretrieve(masterdata_source, '/content/masterdata.bin.e')
    size_kb = os.path.getsize('/content/masterdata.bin.e') / 1024
    print(f'  Downloaded {size_kb:.1f} KB')

print('\nPatching ...')
_run(f'{sys.executable} /content/scripts-jp/masterdata/patch_masterdata.py '
     f'--input /content/masterdata.bin.e --output /content/20240404193219.bin.e',
     'patch_masterdata')

size_kb = os.path.getsize('/content/20240404193219.bin.e') / 1024
print(f'\nDone! Patched master data: {size_kb:.1f} KB')

#files.download('/content/20240404193219.bin.e')
